# 00 — Connect credentials

Save your Scopus API key without leaving the notebook. The key is written to a
`.env` file, never into this notebook: the field below is a password input, it is
cleared after every save, and nothing you type is kept in the notebook's saved
state.

**Where the key goes.** In a notebook the working directory is often
`notebooks/`, not the repository root, while the `prismabib` command reads `.env`
from the directory you run it in (normally the repository root). This notebook
therefore looks upward from its working directory for the repository root (the
first folder holding `.git`) and points the panel at that folder's `.env`. The
path it will use is printed on the panel, under "File checked".

**Warning.** The panel only writes when git confirms that the `.env` file is
ignored. If it is not, the save is refused and nothing is written.

In [ ]:
import os
from pathlib import Path

from prismabib.errors import ConfigError
from prismabib.project import Project


def repository_root(start: Path) -> Path:
    """The nearest folder at or above ``start`` that holds ``.git``, else ``start``."""
    for folder in (start, *start.parents):
        if (folder / ".git").exists():
            return folder
    return start


ENV_FILE = repository_root(Path.cwd().resolve()) / ".env"

# Set PRISMABIB_NOTEBOOK_SLUG to tie the panel to a project (it then says when the
# project's mode does not use Scopus). Without it, or when the project cannot be
# opened, the panel is project-less and always writable.
SLUG = os.environ.get("PRISMABIB_NOTEBOOK_SLUG")
project = None
if SLUG:
    try:
        project = Project.open(SLUG)
    except ConfigError as error:
        print(error)
print(f"credentials file: {ENV_FILE}")

## Connect Scopus

Paste the key and press **Save key**. Run `prismabib credentials test scopus`
afterwards to check it with one request.

In [ ]:
import panel as pn

from prismabib.credentials_ui import ConnectScopusPanel

pn.extension()
panel = ConnectScopusPanel(project, env_file=ENV_FILE)
panel.view()